In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# ============================
# IMPORT MODEL
# ============================
import photop as model


# ============================
# LOAD DATA
# ============================
data = pd.read_csv("selected_omega_dataset.csv")

# Adjust column names if needed
W_data = data["W"].values
sigma_data = data["dsigdt (mu barn/GeV^2)"].values
err_data = data["dsigdt error (mu barn/GeV^2)"].values
t_data = -data["t"].values


# ============================
# MODEL WRAPPER
# ============================
def sigma_model(W,params):
    return model.sig_omega(W,params)


# ============================
# CHI-SQUARED FUNCTION
# ============================
def chi_squared(params):
    model_vals = np.array([sigma_model(W, params) for W in W_data])
    return np.sum(((sigma_data - model_vals) / err_data)**2)

# ============================
# INITIAL PARAMETERS
# ============================
initial_params = np.array([
    13.0, 6.0,        # g_pi_NN, g_eta_NN
    1.0, 1.0,         # Lambda_pi_NN, Lambda_eta_NN
    1.0, 1.0,         # Lambda_pi_gam_omega, Lambda_eta_gam_omega
    1.0, 1.0,         # Lambda_pi_gam_phi, Lambda_eta_gam_phi
    2.0, 2.0,         # b_P_omega_omega, b_P_phi_phi
    4.0, 4.0,         # B_omega, B_phi
    0.25,             # alpha_Pom_prime
    0.1, 0.1          # a_P_omega_omega, a_P_phi_phi
])


# ============================
# FITTING
# ============================
print("Starting fit...")

result = minimize(
    chi_squared,
    initial_params,
    method='Nelder-Mead',
    options={'maxiter': 5000, 'disp': True}
)

best_params = result.x

print("\nBest-fit parameters:")
for i, val in enumerate(best_params):
    print(f"param[{i}] = {val}")


# ============================
# GENERATE FIT CURVE
# ============================
W_fit = np.linspace(min(W_data), max(W_data), 200)
sigma_fit = np.array([sigma_model(W, best_params) for W in W_fit])


# ============================
# PLOT RESULTS
# ============================
plt.figure()

plt.errorbar(
    W_data,
    sigma_data,
    yerr=err_data,
    fmt='o',
    label="Data"
)

plt.plot(
    W_fit,
    sigma_fit,
    label="Model Fit"
)

plt.xlabel(" W (GeV)")
plt.ylabel("Cross section sigma (mu barn/GeV^2)")
plt.title("Cross section sigma vs W ")
plt.legend()

plt.show()



Starting fit...


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import emcee
import corner
import photop as model

# ============================
# LOAD DATA
# ============================
data = pd.read_csv("selected_omega_dataset.csv")
W_data = data["W"].values
sigma_data = data["dsigdt (mu barn/GeV^2)"].values
err_data = data["dsigdt error (mu barn/GeV^2)"].values

# ============================
# MCMC PROBABILITY FUNCTIONS
# ============================

def log_likelihood(params, W, sigma, err):
    """ Standard Gaussian Log-Likelihood """
    model_vals = np.array([model.sig_omega(w, params) for w in W])
    # This is essentially -0.5 * Chi-Squared
    return -0.5 * np.sum(((sigma - model_vals) / err)**2)

def log_prior(params):
    """ 
    Define your parameter bounds. 
    MCMC needs boundaries to prevent 'walking' into non-physical values.
    """
    # Example: Ensure all Lambdas and couplings are positive
    if np.all(params > 0) and np.all(params < 50): 
        return 0.0
    return -np.inf

def log_probability(params, W, sigma, err):
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(params, W, sigma, err)

# ============================
# INITIALIZATION
# ============================
# Use your previous 'best_params' or initial_params
initial_guess = np.array([13.0, 6.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.0, 2.0, 4.0, 4.0, 0.25, 0.1, 0.1])
n_params = len(initial_guess)
n_walkers = 32  # Number of chains (usually 2x to 10x the number of params)
n_steps = 2000  # Total length of the chain

# Initialize walkers in a tiny Gaussian ball around the initial guess
pos = initial_guess + 1e-4 * np.random.randn(n_walkers, n_params)

# ============================
# RUN MCMC
# ============================
print("Starting MCMC Sampling...")
sampler = emcee.EnsembleSampler(n_walkers, n_params, log_probability, args=(W_data, sigma_data, err_data))
sampler.run_mcmc(pos, n_steps, progress=True)

# ============================
# PROCESS SAMPLES
# ============================
# Discard the 'burn-in' (initial phase where walkers find the peak)
samples = sampler.get_chain(discard=200, thin=15, flat=True)

# Calculate Median and Uncertainties (16th, 50th, 84th percentiles)
best_fit_mcmc = np.median(samples, axis=0)

print("\nMCMC Results (Median values):")
for i, val in enumerate(best_fit_mcmc):
    print(f"param[{i}] = {val:.4f}")

# ============================
# VISUALIZATION
# ============================

# 1. Corner Plot (Correlations)
fig = corner.corner(samples, labels=[f"p{i}" for i in range(n_params)], truths=best_fit_mcmc)
plt.show()

# 2. Fit Curve with Uncertainty Bands
plt.figure()
W_fit = np.linspace(min(W_data), max(W_data), 200)

# Plot a few random samples from the chain to show uncertainty
inds = np.random.randint(len(samples), size=100)
for ind in inds:
    sample_params = samples[ind]
    plt.plot(W_fit, [model.sig_omega(w, sample_params) for w in W_fit], color="gray", alpha=0.1)

plt.errorbar(W_data, sigma_data, yerr=err_data, fmt='o', label="Data", color="red")
plt.plot(W_fit, [model.sig_omega(w, best_fit_mcmc) for w in W_fit], label="MCMC Median Fit", color="blue")
plt.xlabel("W (GeV)")
plt.ylabel("Cross section")
plt.legend()
plt.show()

Starting MCMC Sampling...


You must install the tqdm library to use progress indicators with emcee
